# Supermarket Sales — Practice

1,000 transactions from 3 branches in Myanmar (Jan–Mar 2019). 17 columns, no nulls, no duplicates.

**Quirks to know:**
- `Date` and `Time` are strings — convert them yourself.
- `Tax 5%` and `gross income` are identical columns.
- `gross margin percentage` is a constant (4.76%) — useless as a feature.
- `Total = cogs + Tax 5%`, and `cogs = Unit price × Quantity`.

Try each exercise yourself before searching for help. Exercises get harder as you go.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/supermarket_sales.csv")
df.head()

,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Total,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating
0,750-67-8428,A,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,1/5/2019,13:08,Ewallet,522.83,4.761905,26.1415,9.1
1,226-31-3081,C,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.8200,80.2200,3/8/2019,10:29,Cash,76.40,4.761905,3.8200,9.6
2,631-41-3108,A,Yangon,Normal,Male,Home and lifestyle,46.33,7,16.2155,340.5255,3/3/2019,13:23,Credit card,324.31,4.761905,16.2155,7.4
3,123-19-1176,A,Yangon,Member,Male,Health and beauty,58.22,8,23.2880,489.0480,1/27/2019,20:33,Ewallet,465.76,4.761905,23.2880,8.4
4,373-73-7910,A,Yangon,Normal,Male,Sports and travel,86.31,7,30.2085,634.3785,2/8/2019,10:37,Ewallet,604.17,4.761905,30.2085,5.3


## Level 1 — Cleaning & basics
1. Convert `Date` to datetime. Combine `Date` + `Time` into one `datetime` column.
2. Add `month`, `weekday`, and `hour` columns.
3. Verify the formulas: does `Unit price * Quantity` equal `cogs`? Does `cogs * 1.05` equal `Total`?
4. Drop the redundant columns (which ones, and why?).
5. Rename columns to snake_case (`Unit price` → `unit_price`).

In [2]:
df.shape
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Invoice ID               1000 non-null   object 
 1   Branch                   1000 non-null   object 
 2   City                     1000 non-null   object 
 3   Customer type            1000 non-null   object 
 4   Gender                   1000 non-null   object 
 5   Product line             1000 non-null   object 
 6   Unit price               1000 non-null   float64
 7   Quantity                 1000 non-null   int64  
 8   Tax 5%                   1000 non-null   float64
 9   Total                    1000 non-null   float64
 10  Date                     1000 non-null   object 
 11  Time                     1000 non-null   object 
 12  Payment                  1000 non-null   object 
 13  cogs                     1000 non-null   float64
 14  gross margin percentage  

,Unit price,Quantity,Tax 5%,Total,cogs,gross margin percentage,gross income,Rating
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.00000,1000.000000,1000.000000,1000.00000
mean,55.672130,5.510000,15.379369,322.966749,307.58738,4.761905,15.379369,6.97270
std,26.494628,2.923431,11.708825,245.885335,234.17651,0.000000,11.708825,1.71858
min,10.080000,1.000000,0.508500,10.678500,10.17000,4.761905,0.508500,4.00000
25%,32.875000,3.000000,5.924875,124.422375,118.49750,4.761905,5.924875,5.50000
50%,55.230000,5.000000,12.088000,253.848000,241.76000,4.761905,12.088000,7.00000
75%,77.935000,8.000000,22.445250,471.350250,448.90500,4.761905,22.445250,8.50000
max,99.960000,10.000000,49.650000,1042.650000,993.00000,4.761905,49.650000,10.00000


In [3]:
df["Date"] = pd.to_datetime(df["Date"])

df.info()
df["Date"].min(), df["Date"].max()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Invoice ID               1000 non-null   object        
 1   Branch                   1000 non-null   object        
 2   City                     1000 non-null   object        
 3   Customer type            1000 non-null   object        
 4   Gender                   1000 non-null   object        
 5   Product line             1000 non-null   object        
 6   Unit price               1000 non-null   float64       
 7   Quantity                 1000 non-null   int64         
 8   Tax 5%                   1000 non-null   float64       
 9   Total                    1000 non-null   float64       
 10  Date                     1000 non-null   datetime64[ns]
 11  Time                     1000 non-null   object        
 12  Payment                  1000 non-n

(Timestamp('2019-01-01 00:00:00'), Timestamp('2019-03-30 00:00:00'))

In [6]:
df["month"] = df["Date"].dt.month
df["month"].value_counts()

month
1    352
3    345
2    303
Name: count, dtype: int64

In [10]:
df["weekday"] = df["Date"].dt.day_name() 
df["weekday"].value_counts()

weekday
Saturday     164
Tuesday      158
Wednesday    143
Friday       139
Thursday     138
Sunday       133
Monday       125
Name: count, dtype: int64

In [13]:
df["hour"] = pd.to_datetime(df["Time"], format="%H:%M").dt.hour
df["hour"].value_counts().sort_index()

hour
10    101
11     90
12     89
13    103
14     83
15    102
16     77
17     74
18     93
19    113
20     75
Name: count, dtype: int64

In [14]:
df["Unit price"] * df["Quantity"]

0      522.83
1       76.40
2      324.31
3      465.76
4      604.17
        ...  
995     40.35
996    973.80
997     31.84
998     65.82
999    618.38
Length: 1000, dtype: float64

In [15]:
check = df[["Unit price", "Quantity", "cogs"]].copy()
check["my_cogs"] = df["Unit price"] * df["Quantity"]
check.head()

,Unit price,Quantity,cogs,my_cogs
0,74.69,7,522.83,522.83
1,15.28,5,76.40,76.40
2,46.33,7,324.31,324.31
3,58.22,8,465.76,465.76
4,86.31,7,604.17,604.17


In [16]:
(check["my_cogs"] - check["cogs"]).abs().max()

1.1368683772161603e-13

In [17]:
(df["cogs"] * 1.05 - df["Total"]).abs().max()

1.1368683772161603e-13

In [18]:
(df["Tax 5%"]-df["gross income"]).abs().max()

0.0

In [20]:
df = df.drop(columns=["gross margin percentage", "gross income"])
df.shape

(1000, 18)

In [21]:
df.columns

Index(['Invoice ID', 'Branch', 'City', 'Customer type', 'Gender',
       'Product line', 'Unit price', 'Quantity', 'Tax 5%', 'Total', 'Date',
       'Time', 'Payment', 'cogs', 'Rating', 'month', 'weekday', 'hour'],
      dtype='object')

In [22]:
df.columns = df.columns.str.lower()
df.columns

Index(['invoice id', 'branch', 'city', 'customer type', 'gender',
       'product line', 'unit price', 'quantity', 'tax 5%', 'total', 'date',
       'time', 'payment', 'cogs', 'rating', 'month', 'weekday', 'hour'],
      dtype='object')

In [24]:
df.columns = df.columns.str.lower().str.replace(" ", "_")
df.columns

Index(['invoice_id', 'branch', 'city', 'customer_type', 'gender',
       'product_line', 'unit_price', 'quantity', 'tax_5%', 'total', 'date',
       'time', 'payment', 'cogs', 'rating', 'month', 'weekday', 'hour'],
      dtype='object')

## Level 2 — Aggregation (groupby)
6. Total revenue per branch/city. Which is highest?
7. Revenue and average quantity per product line, sorted.
8. Most popular payment method in each city.
9. Do members spend more per transaction than normal customers?
10. Average rating per product line — which is lowest?

## Level 3 — Time & pivots
11. Plot daily revenue over the 3 months. Any trend?
12. Which hour of the day has the most sales? Which weekday?
13. Pivot table: rows = product line, columns = gender, values = total revenue.
14. Month-over-month revenue growth (%) per branch.
15. Heatmap of revenue by weekday × hour.

## Level 4 — Interview-style questions
16. Is there a statistically significant difference in `Total` between members and normal customers? (t-test)
17. Is `Rating` correlated with `Total` or `Unit price`? What does that tell you?
18. Is product line choice independent of gender? (chi-square test)
19. Write the same queries as Level 2 in SQL (load the CSV into `sqlite3` with `df.to_sql`).
20. Write a 5-bullet summary for a store manager: what should they do differently?